In [0]:
import pyspark
from pyspark.sql.types import *
import pyspark.sql.functions as f
from pyspark.sql import SparkSession, Row, DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from pyspark.sql.functions import when
from typing import Union, Optional, List
from dataclasses import dataclass
from pyspark.sql.types import IntegerType
from functools import reduce
from datetime import timedelta
from pyspark.sql.functions import broadcast
from pyspark.sql.functions import when, lit

#import
#from pls_common_data_store import pls_data_store
#pds = pls_data_store()

#ignore strange depreciation warnings
from warnings import simplefilter 
simplefilter(action='ignore', category=DeprecationWarning)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.shuffle.partitions","auto")
spark.conf.get("spark.sql.shuffle.partitions")

# spark.conf.set("spark.databricks.queryWatchdog.maxQueryTasks", "50000000")

#### Absolute Metrics Calculation: XCM

In [0]:
# Pre-pivoted closed loop data pulled from closed_loop_campaign_summary notebook
closed_loop_prepivot = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/closed_loop_summary_tab_INTERMEDIATE.csv')
closed_loop_prepivot = closed_loop_prepivot.filter(f.col('camp_start_date') >= '2023-01-01')
closed_loop_prepivot.display()

# Metadata pulls from KPM
mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')
mda = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/dashboard/campaign/version=v2/source=azure')
points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
mmoi= spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/MEDIA_MEAS_OFFER_INFO')
new_th = spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/TARGET_HISTORY')
old_th = spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/bullseye/TARGET_HISTORY_FULL_20251015/').withColumnRenamed('ehhn', 'hshd_code').select('TARGET_ID', 'HSHD_CODE', 'PRIORITY', 'TEST_CONTROL_ID', 'OFFER_ID', 'DECILE', 'SCORE')
target_history = new_th.union(old_th)
mhtv = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped')
redemptions = spark.read.parquet(f'abfss://acds@sa8451posprd.dfs.core.windows.net/transaction_coupon_fct')
downloads = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/intermediate/engagements/coupon_downloads/')

# Rem sse push barcodes automated
rem_sse_push_offer_codes = spark.read.parquet(f'abfss://tmkrprtnrs-users@sa8451krprtnrdev.dfs.core.windows.net/a136627/META_DATA_INFO')

In [0]:
abs_offsite_xcm = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_xcm_disp')
abs_offsite_xcm.display()

In [0]:
# filter current dashboard to only include XCM
closed_loop_prepivot_xcm_disp = closed_loop_prepivot.filter(
    (f.col("campaign_type") == "DISPLAY_AD") | (f.col("campaign_type") == "XCM") | (f.col("campaign_type") == "EMOD") | 
    (f.col("campaign_type") == "OLV")| (f.col("campaign_type") == "PINT") 
)
closed_loop_prepivot_xcm_disp.display()

In [0]:
rem_sse_push_offer_codes.display()
rem_sse_push_offer_codes_small = rem_sse_push_offer_codes.select("COUPON_BARCODE", "KPM_DUPLICATED_ID", "KPM_PROJECT_ID", "CHANNEL")
# Make into list of tuples, similar to offsite 
barcode_id_camptype_tuple = [(row[0], row[1], row[2], row[3]) for row in rem_sse_push_offer_codes_small.collect()]
barcode_id_camptype_tuple

In [0]:
data = [ 
    (800000202496, 168460, 'DISPLAY_AD'),
    (800000201991, 168460, 'DISPLAY_AD'),
    (800000201992, 168460, 'DISPLAY_AD'),
    (800000202495, 168460, 'DISPLAY_AD'),

    (800000202496, 168462, 'PINT'),
    (800000201991, 168462, 'PINT'),
    (800000201992, 168462, 'PINT'),
    (800000202495, 168462, 'PINT'),

    (800000202496, 168461, 'PRV'),
    (800000201991, 168461, 'PRV'),
    (800000201992, 168461, 'PRV'),
    (800000202495, 168461, 'PRV'),
]


schema = ["coupon_barcode", "campaign_id", "campaign_type"]
coupon_df = spark.createDataFrame(data, schema)
coupon_df.display()

campaign_ids = coupon_df.select('campaign_id').distinct().rdd.flatMap(lambda x: x).collect()
print(campaign_ids)

In [0]:
# Replace this with read-in Excel file containing coupon codes, campaign IDs, and campaign type
'''
data = [ 
    (800000200163, 163164, 'DISPLAY_AD'),
    (800000200162, 163164, 'DISPLAY_AD')
    
    # 2025 Spring Easter Mothers Day
    #(800000538427, 131511, 'XCM'),
    #(800000538426, 131511, 'XCM'),
    #(800000538459, 131511, 'XCM'),
    #(800000538460, 131511, 'XCM'),
    #(800000727746, 131511, 'XCM'),
    #(800000727744, 131511, 'XCM'),
    #(800000727743, 131511, 'XCM'),
    #(800000727745, 131511, 'XCM'),
    #(800000786477, 131511, 'XCM'),
    #(800000786476, 131511, 'XCM'),
    #(800000786479, 131511, 'XCM'),
    #(800000786478, 131511, 'XCM'), ]

    #(800000176164, 153839, 'XCM'),
    #(800000175826, 153839, 'XCM'),
    #(800000176170, 153839, 'XCM'),
    #(800000176075, 153839, 'XCM'),
    #(800000176162, 153839, 'XCM'),
    #(800000176161, 153839, 'XCM'),
    #(800000175835, 153839, 'XCM'),
    #(800000176087, 153839, 'XCM'),
]

    # 2025 HOLIDAY XCM
    (800000176164, 153839, 'XCM'),
    (800000175826, 153839, 'XCM'),
    (800000176170, 153839, 'XCM'),
    (800000176075, 153839, 'XCM'),
    (800000176162, 153839, 'XCM'),
    (800000176161, 153839, 'XCM'),
    (800000175835, 153839, 'XCM'),
    (800000176087, 153839, 'XCM'),
    # 2025 WInter 4x DISP
    (800000192179, 161613, 'DISPLAY_AD'),
    (800000192181, 161613, 'DISPLAY_AD'),
    (800000192180, 161613, 'DISPLAY_AD'),
    (800000192838, 161613, 'DISPLAY_AD'),

    # Campaigns w No Offer
    (000000000000, 147400, 'XCM'),
    (000000000000, 136421, 'XCM'),
    # 2025 Fathers Day Summer
    (800000835408, 139466, 'XCM'),
    (800000835410, 139466, 'XCM'),
    (800000831624, 139466, 'XCM'),
    (800000835409, 139466, 'XCM'),
    (800000132655, 139466, 'XCM'),
    (800000132654, 139466, 'XCM'),
    (800000132656, 139466, 'XCM'),
    (800000131272, 139466, 'XCM'),
    # 2025 BTS
    (800000138292, 141800, 'XCM'),
    (800000138286, 141800, 'XCM'),
    (800000138299, 141800, 'XCM'),
    (800000138300, 141800, 'XCM'),
    # 2025 Valentines Day
    (800000531437, 128642, 'DISPLAY_AD'),
    (800000531431, 128642, 'DISPLAY_AD'),
    (800000531433, 128642, 'DISPLAY_AD'),
    (800000531432, 128642, 'DISPLAY_AD'),
    # 2025 Halloween
    (800000164400, 155120, 'DISPLAY_AD'), 
    (800000164397, 155120, 'DISPLAY_AD'), 
    (800000164398, 155120, 'DISPLAY_AD'), 
    (800000164399, 155120, 'DISPLAY_AD'),
    # 2025 Fall 4X Dinner Game
    (800000149690, 151089, 'XCM'),
    (800000149688, 151089, 'XCM'),
    (800000149325, 151089, 'XCM'),
    (800000149652, 151089, 'XCM'),
    (800000150177, 151089, 'XCM'),
    (800000150172, 151089, 'XCM'),
    (800000150183, 151089, 'XCM'),
    (800000150201, 151089, 'XCM'),
    # 2024 Easter Q1
    (800000283555, 85399, 'XCM'),
    (800000283554, 85399, 'XCM'),
    # 2024 BTS Q2
    (800000597889, 103442, 'XCM'),
    (800000597689, 103442, 'XCM'),
    (800000619889, 103442, 'XCM'),
    (800000620289, 103442, 'XCM'),
    # 2024 Holiday Q3
    (800000616531, 116942, 'XCM'),
    (800000616534, 116942, 'XCM'),
    (800000616535, 116942, 'XCM'),
    (800000616530, 116942, 'XCM'),
    # 2024 Mothers Day Fathers Day
    (800000320450, 91440, 'XCM'),
    (800000320449, 91440, 'XCM'),
    (800000380740, 91440, 'XCM'),
    (800000380739, 91440, 'XCM'),
    (800000364723, 91440, 'XCM'),
    (800000365249, 91440, 'XCM'), 


schema = ["coupon_barcode", "campaign_id", "campaign_type"]
coupon_df = spark.createDataFrame(data, schema)
coupon_df.display()

campaign_ids = coupon_df.select('campaign_id').distinct().rdd.flatMap(lambda x: x).collect()
print(campaign_ids)

'''

In [0]:
print(campaign_ids)

mmci.filter(f.col('kpm_duplicated_id') == "168460").display()

In [0]:
# Working Cost: Camp Cost * Multiplier (depending on channel type)
mmci_xcm_display_working_cost = (mmci.filter(f.col('kpm_project_id').isin(campaign_ids))
    .withColumn('multiplier',
        f.when(f.col('channel') == 'Display Ad', f.lit(0.3520))
         .when(f.col('channel') == 'Email Module', f.lit(0.015))
         .when(f.col('channel') == 'Pandora', f.lit(0.741))
         .when(f.col('channel') == 'Pinterest', f.lit(0.663))
         .when(f.col('channel') == 'Pre-Roll Video', f.lit(.3960))
         .when(f.col('channel') == 'Push Notifications', f.lit(0.038))
         .when(f.col('channel') == 'Roku', f.lit(0.90))
         .when(f.col('channel') == 'Native', f.lit(1))
         .when(f.col('channel') == 'Single Subject Email', f.lit(0.3390))
         .when(f.col('channel') == 'Targeted Digital Coupon', f.lit(0.141))
         .otherwise(f.lit(0))
    )
    .withColumn('working_cost', (f.col('TOT_COST') * f.col('multiplier'))) # Keep as double for now
    .groupBy('KPM_PROJECT_ID')
    .agg(
        f.max('CAMP_START_DATE').alias('camp_start_date'),
        f.max('CAMP_END_DATE').alias('camp_end_date'),
        f.sum('TOT_COST').alias('camp_cost_mmci'),
        f.sum('working_cost').alias('working_cost'),
    )
)

mmci_xcm_display_working_cost = mmci_xcm_display_working_cost.withColumnRenamed('KPM_DUPLICATED_ID', 'campaign_id')
mmci_xcm_display_working_cost.display()

In [0]:
# Pulling coupons from Excel / Temp DF
coupon_barcodes_agg = (coupon_df
    .withColumn('coupon_barcode', f.col('coupon_barcode').cast('string'))
    .withColumn('redemption_barcode', f.lpad(f.col('coupon_barcode'), 13, '0'))
    .groupBy('campaign_id').agg(
        f.collect_set('coupon_barcode').alias('coupon_barcodes'),
        f.collect_set('redemption_barcode').alias('redemption_barcodes')
    )
    .withColumnRenamed('campaign_id', 'kpm_duplicated_id')
)

mmci_xcm_display_working_cost_barcodes = mmci_xcm_display_working_cost.join(
    coupon_barcodes_agg,
    mmci_xcm_display_working_cost.KPM_PROJECT_ID == coupon_barcodes_agg.kpm_duplicated_id,
    'inner'
)
display(mmci_xcm_display_working_cost_barcodes)

In [0]:
measurement_path = 'abfss://measure@sa8451camprd.dfs.core.windows.net'
version = 'v2'
source = 'azure'

# 2025-05-28 added this numeric check to mda campaign ids due to a bad backend change on their end
mda = (
    spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/dashboard/campaign/version=v2/source=azure')
    .filter(f.col("campaign_id").rlike("^[0-9]+$"))
).cache()

# Get all job ids for xcm and display ad campaigns (using campaign id) and make into a list
job_id_rows = (mda.filter(f.col('campaign_id').isin(campaign_ids)) # chek
               #.filter(f.col('downstream_ready') == 'true')
               .select('campaign_id', 'job_id', 'campaign_type').distinct().collect())

# Create a dictionary where the key is campaign_id and the value is a tuple containing both job_id and campaign_type (Need to fetch campaign type to use in path call)
camp_id_job_type_dict = {
    row['campaign_id']: {'job_id': row['job_id'], 'camp_type': row['campaign_type']} 
    for row in job_id_rows
}

print(camp_id_job_type_dict)

In [0]:
# Loop through key value pair of kpm id + job id (to use in projection path dynamically), loop needed here i think because paths are different per kpm id 
projection_results = []
for camp_id, camp_info in camp_id_job_type_dict.items():
  job_id = camp_info['job_id']
  camp_type = camp_info['camp_type'] 

  # Filters different depending on campaign type - XCM (filters are shumailas logic)
  if camp_type == "XCM":
    print(camp_id, camp_type, job_id)
    projection = (spark.read.parquet(
      f'{measurement_path}/reports/projection/version={version}/source={source}/campaign_type={camp_type}/campaign_id={camp_id}/job_id={job_id}')
        .filter(~f.col('product_group').like('%_online%'))
        .filter(~f.col('product_group').like('%_in_store%'))
        .filter(f.col('segment') == 'ALL')
        .withColumn('scale_factor', f.round(f.col('projection_factor').cast('double'), 2))
        .withColumn('kpm_duplicated_id', f.lit(camp_id))
        .select("kpm_duplicated_id", "scale_factor", "projection_factor")
    )
    projection_results.append(projection)

  # Filters different depending on campaign type - DISPLAY (filters are Shumailas logic) or EMOD (?)
  elif camp_type != "XCM":
    print(camp_id, camp_type, job_id)
    projection = (spark.read.parquet(
      f'{measurement_path}/reports/media_metrics/campaign/version={version}/source={source}/campaign_type={camp_type}/campaign_id={camp_id}/job_id={job_id}')
        .filter(f.col('modality') == 'All Modalities')
        .filter(f.col('segment') == 'ALL')
        .filter(f.col('sub_segment') == 'ALL')
        #.filter(f.col('product_group') == 'Third Party And Open Loop 010225')
        .filter(f.col('rom') == 'Kroger Only').filter(f.col('metric') == 'scale_factor')
        .withColumnRenamed('value', 'projection_factor').select('projection_factor').distinct()
        .withColumn('scale_factor', f.round(f.col('projection_factor').cast('double'), 2))
        .withColumn("kpm_duplicated_id", f.lit(camp_id))
        .select("kpm_duplicated_id", "scale_factor", "projection_factor")
    )
    projection_results.append(projection)

# Union All Data to obtain scale factor + projection factor per kpm duplicated id
projection_all = reduce(lambda df1, df2: df1.union(df2), projection_results).dropDuplicates()

display(projection_all)

In [0]:
# Get Test HHs per campaign, along with # Distinct Test Households for the campaign

hhs_cont_test_list = [] 
distinct_test_hhs_list = [] 

for camp_id, camp_info in camp_id_job_type_dict.items():
    job_id = camp_info['job_id']
    camp_type = camp_info['camp_type'] 
    print(job_id, camp_type)

    '''
    # HH matching based on camp_type XCM
    if camp_type != "DISPLAY_AD":
        hhs_match_original = spark.read.parquet(
            f'{measurement_path}/intermediate/hh_matching/match/campaign_type={camp_type}/campaign_id={camp_id}/job_id={job_id}'
        ).withColumn('kpm_duplicated_id', f.lit(camp_id))
        hh_cont = hhs_match_original.select('kpm_duplicated_id', f.col('cont_hh').alias('ehhn')).withColumn('hhgroup', f.lit('CONT'))
        hh_test = hhs_match_original.select('kpm_duplicated_id', f.col('test_hh').alias('ehhn')).withColumn('hhgroup', f.lit('TEST'))

        hh_cont_test = hh_cont.union(hh_test).withColumn('kpm_duplicated_id', f.lit(camp_id))
        distinct_test_hh = hh_cont_test.filter(f.col('hhgroup') == 'TEST').groupBy('hhgroup', 'kpm_duplicated_id').agg(f.countDistinct('ehhn').alias('distinct_test_hhs'))

        hhs_cont_test_list.append(hh_cont_test)
        distinct_test_hhs_list.append(distinct_test_hh)
    '''

    # HH matching based on camp_type DISPLAY_AD (Change this for other Channel Types)
    if camp_type != "XCM":
        hhs_match_original = spark.read.parquet(
            f'abfss://measure@sa8451camprd.dfs.core.windows.net/inputs/households/campaign_type={camp_type}/campaign_id={camp_id}/job_id={job_id}'
            #f'abfss://measure@sa8451camprd.dfs.core.windows.net/inputs/households/campaign_type=DISPLAY_AD/campaign_id=128642/job_id=2025-05-12-1747078257-MAEKM'
        ).withColumn('kpm_duplicated_id', f.lit(camp_id))
        hhs_cont = hhs_match_original.filter(f.col("hhgroup") == "CONT").select('kpm_duplicated_id', 'hshd_code', 'hhgroup').withColumnRenamed('hshd_code', 'ehhn')
        hhs_test = hhs_match_original.filter(f.col("hhgroup") == "TEST").select('kpm_duplicated_id', 'hshd_code', 'hhgroup').withColumnRenamed('hshd_code', 'ehhn')

        hh_cont_test = hhs_cont.union(hhs_test).withColumn('kpm_duplicated_id', f.lit(camp_id))
        distinct_test_hh = hh_cont_test.filter(f.col('hhgroup') == 'TEST').groupBy('hhgroup', 'kpm_duplicated_id').agg(f.countDistinct('ehhn').alias('distinct_test_hhs'))

        hhs_cont_test_list.append(hh_cont_test)
        distinct_test_hhs_list.append(distinct_test_hh)

if hhs_cont_test_list:
    hhs_cont_test = reduce(lambda df1, df2: df1.unionByName(df2), hhs_cont_test_list)
    hhs_cont_test.persist()
    
if distinct_test_hhs_list:
    distinct_test_hhs = reduce(lambda df1, df2: df1.unionByName(df2), distinct_test_hhs_list)

distinct_test_hhs = distinct_test_hhs.dropDuplicates(['kpm_duplicated_id', 'hhgroup']) 
distinct_test_hhs.display()

In [0]:
# Join points detail with households
points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
points_detail_ehhns = (points_detail
    .join(broadcast(hhs_cont_test), on='ehhn', how='inner'))

display(points_detail_ehhns)
# Making sure all campaign_ids have HH points detail
distinct_kpm_ids = points_detail_ehhns.select('kpm_duplicated_id').distinct()
display(distinct_kpm_ids)

In [0]:
# Join Points Detail w/ Dashboard and Select only coupon and redemption barcode offers
points_detail_ehhns_barcodes = (points_detail_ehhns
    .withColumn('trn_dt', f.to_date('trn_dt', 'yyyyMMdd'))
    .join(mmci_xcm_display_working_cost_barcodes.select('kpm_duplicated_id', 'coupon_barcodes', 'redemption_barcodes', 'camp_start_date', 'camp_end_date'), 
          on = "kpm_duplicated_id",
          how='inner')
    .filter(
        f.array_contains(f.col('coupon_barcodes'), f.col('offer')) |
        f.array_contains(f.col('redemption_barcodes'), f.col('offer'))
    )
)

# Filter transaction date within camp_start and camp_end date
points_detail_ehhn_barcodes_filtered = (points_detail_ehhns_barcodes
    .filter(
        (f.col('trn_dt') >= f.to_date(f.col('camp_start_date'), 'yyyyMMdd')) &
        (f.col('trn_dt') <= f.to_date(f.col('camp_end_date'), 'yyyyMMdd'))
    )
)

points_detail_ehhn_barcodes_filtered.display()

In [0]:
# Calculate aggregate points detail (earned and redeemed) CONTROL: total_hhs,	distinct_hhs, total_points_earned, total_points_redeemed, 
# distinct_test_hhs. total_measured_hhs	earn_per_hh	measured_hhs_earned
points_detail_ehhn_barcodes_filtered_CONT = (points_detail_ehhn_barcodes_filtered
    .filter(f.col('hhgroup') == 'CONT')
    .groupBy('hhgroup', 'kpm_duplicated_id')
    .agg(
      f.count('ehhn').alias('total_hhs'),
      f.countDistinct('ehhn').alias('distinct_hhs'), 
      f.sum('points_earned').alias('total_points_earned'), 
      f.sum('points_redeemed').alias('total_points_redeemed')
    )
    .join(
      distinct_test_hhs.select('kpm_duplicated_id', 'distinct_test_hhs'), on = 'kpm_duplicated_id', how = 'inner'
    )
    .join(
      coupon_df.select(col("campaign_id").alias("kpm_duplicated_id"), "campaign_type"), on = "kpm_duplicated_id", how = "left"
    )
    .withColumn(
      'total_measured_hhs', f.col('distinct_test_hhs')
    )
    # Shumaila's logic - denominator is different depending on camp_type
    .withColumn(
      'earn_per_hh', when(f.col("campaign_type") == "XCM", f.col('total_points_earned') / f.col('total_measured_hhs')
      ).otherwise(
        f.col('total_points_earned') / f.col('total_measured_hhs')
      )
    )
    .withColumn('measured_hhs_earned', f.col('earn_per_hh') * f.col('total_measured_hhs'))
    .dropDuplicates()
)

points_detail_ehhn_barcodes_filtered_CONT.display()

In [0]:
# Calculate aggregate points detail (earned and redeemed) TEST: total_hhs,	distinct_hhs, total_points_earned, total_points_redeemed, 
# distinct_test_hhs. total_measured_hhs	earn_per_hh	measured_hhs_earned
points_detail_ehhn_barcodes_filtered_TEST = (points_detail_ehhn_barcodes_filtered
    .filter(f.col('hhgroup') == 'TEST')
    .groupBy('hhgroup', 'kpm_duplicated_id')
    .agg(
      f.count('ehhn').alias('total_hhs'),
      f.countDistinct('ehhn').alias('distinct_hhs'), 
      f.sum('points_earned').alias('total_points_earned'), 
      f.sum('points_redeemed').alias('total_points_redeemed')
    )
    .join(
      distinct_test_hhs.select('kpm_duplicated_id', 'distinct_test_hhs'), on = 'kpm_duplicated_id', how = 'inner'
    )
    .join(
      coupon_df.select(col("campaign_id").alias("kpm_duplicated_id"), "campaign_type"), on = "kpm_duplicated_id", how = "left"
    )
    .withColumn(
      'total_measured_hhs', f.col('distinct_test_hhs')
    )

    # Shumaila's logic - denominator is different depending on camp_type
    .withColumn(
      'earn_per_hh', when(f.col("campaign_type") == "XCM", f.col('total_points_earned') / f.col('total_measured_hhs')
      # Fixed on 3-2 to make it total points / total measured hhs regardless of tactic                    
      ).otherwise(
        f.col('total_points_earned') / f.col('total_measured_hhs')
      )
    )
    .withColumn('measured_hhs_earned', f.col('earn_per_hh') * f.col('total_measured_hhs'))
    .dropDuplicates()
)

points_detail_ehhn_barcodes_filtered_TEST.display()

In [0]:
# UNION of test and control hh and points numbers
spark.conf.set("spark.sql.shuffle.partitions", "2000") 
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.spill", "true")

points_detail_final = points_detail_ehhn_barcodes_filtered_CONT.union(points_detail_ehhn_barcodes_filtered_TEST)
points_detail_final.display()

In [0]:
mmci_id_data = mmci.select(
    f.col("KPM_DUPLICATED_ID").alias("kpm_duplicated_id"), 
    f.col("KPM_PROJECT_ID"), 
    f.col("CHANNEL").alias("mmci_channel") 
)

closed_loop_prepivoted_revised = closed_loop_prepivot.select(
    "kpm_duplicated_id", "channel", 'sales_uplift_total', 'sales_test_total'
).filter(f.col("campaign_id").isin(campaign_ids)) \
 .withColumn("channel",
    f.when(f.col("channel") == "Display Ad", "DISPLAY_AD")
     .when(f.col("channel") == "Pre-Roll Video", "PRV")
     .when(f.col("channel") == "Pinterest", "PINTEREST")
     .otherwise(f.col("channel"))
 ) \
 .withColumn("channel", f.upper(f.col("channel")))


closed_loop_prepivoted_revised = closed_loop_prepivoted_revised.join(
    mmci_id_data, on=["kpm_duplicated_id"], how="inner"
).drop("kpm_duplicated_id", "channel").dropDuplicates()

display(closed_loop_prepivoted_revised)
#select("kpm_duplicated_id", "KPM_PROJECT_ID", "CHANNEL", "sales_uplift_total", "sales_test_total")

In [0]:
# Preprocessing for efficient cluster performance 
spark.conf.set("spark.sql.shuffle.partitions", "1024")
spark.conf.set("spark.sql.adaptive.enabled", "true")
hhgroups = ['TEST', 'CONT']

# Pivot to have control and test HH and Points #s on new columns
points_detail_final_pivoted_cont_test = (
    points_detail_final
        .groupBy('kpm_duplicated_id')
        .pivot('hhgroup', hhgroups)
        .agg(
            f.first('total_hhs').alias('total_original_hhs'),
            f.first('distinct_hhs').alias('distinct_original_hhs'),
            f.first('total_measured_hhs').alias('total_measured_hhs'),
            f.first('total_points_earned').alias('total_points_earned'),
            f.first('measured_hhs_earned').alias('measured_hhs_earned'),
        )
)

# Add scale factor that we previously calculated - we need this for "inc_total_points_earned_scaled" which is also used for other calcs 
# Bring in campaign cost and working cost that was previously calculated
# Bring in sales uplift and sales test from kpf dashboard
points_detail_final_pivoted_cont_test = (points_detail_final_pivoted_cont_test
    .join(f.broadcast(projection_all), on='kpm_duplicated_id', how='inner')
    .join(f.broadcast(mmci_xcm_display_working_cost_barcodes.select('kpm_duplicated_id','camp_cost_mmci', 'working_cost')), on='kpm_duplicated_id', how='inner')
    .join(f.broadcast(closed_loop_prepivoted_revised.select(f.col('KPM_PROJECT_ID').alias('kpm_duplicated_id'), 'sales_uplift_total', 'sales_test_total')), on='kpm_duplicated_id', how='inner')
)

points_detail_final_pivoted_cont_test.display()

In [0]:
# Revised Calculations from Shumaila's code
points_detail_final_pivoted_cont_test = points_detail_final_pivoted_cont_test.withColumn('sales_test_total', f.col('sales_test_total').cast('double'))

points_detail_final_pivoted_output = (
    points_detail_final_pivoted_cont_test
        .dropDuplicates()
        .withColumn('inc_total_points_earned_scaled', (f.col('TEST_measured_hhs_earned') * f.col('scale_factor')))
        .withColumn('cost_inc_total_points_earned', f.round(f.col('inc_total_points_earned_scaled') * 0.01, 2).cast('double'))
        .withColumn('adj_sales_uplift', f.col('sales_uplift_total').cast('double'))
        .withColumn('adj_sales_total', f.col('sales_test_total').cast('Integer'))
        .withColumn('camp_cost', f.col('camp_cost_mmci'))
        .withColumn('working_cost', f.col('working_cost').cast('Integer'))
        .withColumn('adj_total_cost', f.round((f.col('cost_inc_total_points_earned') + f.col('camp_cost')).cast('double'), 2))
        .withColumn('abs_total_cost', f.round((f.col('cost_inc_total_points_earned') + f.col('working_cost')).cast('double'), 2))
        # Added Redemption Cost and Abs Uplift as Adj - Redemption Cost
        .withColumn('adj_sales_uplift', f.col('sales_uplift_total').cast('double'))
        .withColumn('adj_sales_total', f.col('sales_test_total').cast('double'))
        .withColumn('redemption_cost', f.col('abs_total_cost') - f.col('working_cost'))
        .withColumn('new_sales_uplift_earned', f.round((f.col('adj_sales_uplift') - f.col('redemption_cost')).cast('double'), 2))
        .withColumn('new_sales_test_earned', f.round((f.col('adj_sales_total') - f.col('redemption_cost')).cast('double'), 2))
        .withColumn('adj_iroas', f.round((f.col('adj_sales_uplift') / f.col('adj_total_cost')).cast('double'), 2))
        .withColumn('abs_iroas', f.round((f.col('new_sales_uplift_earned') / f.col('abs_total_cost')).cast('double'), 2))
        .withColumn('adj_aroas', f.round((f.col('adj_sales_total') / f.col('adj_total_cost')).cast('double'), 2))
        .withColumn('abs_aroas', f.round((f.col('new_sales_test_earned') / f.col('abs_total_cost')).cast('double'), 2))
        
)

# Keep only adjusted and abs cols
points_detail_final_pivoted_output.display()

In [0]:
# Appending to file rather than overwriting so that we only need to run automation for NEW campaigns (once absolute #s are calculated for a campaign that has finished measurement, those #s dont change)
points_detail_final_pivoted_output.coalesce(1).write.mode("append").parquet('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_xcm_disp')

#final_result_xcm_disp_df_test = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_xcm_disp')
#final_result_xcm_disp_df_test.display()

#### Processing Final DF

In [0]:
# Processing MCP SSE campaigns + automation
final_result_sse_df_test = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_mcp_sse') \
    .dropDuplicates() \
    .filter(f.col("effective_date") >= "20240101") \
    .withColumnRenamed("abs_sales_uplift_earned", "abs_sales_uplift") \
    .withColumnRenamed("KPM_PROJECT_ID", "kpm_duplicated_id") \
    .withColumnRenamed("redemption_cost", "new_redemption_cost") \
    .select(
        "kpm_duplicated_id", "working_cost", "new_redemption_cost", "adj_sales_uplift", "abs_sales_uplift", "abs_sales_test_earned", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost"
    )

# Make negative #s to 0, since we do not report negative #s.
cols_to_zero_neg = [
    "working_cost", "adj_sales_uplift", "abs_sales_uplift", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost", "abs_sales_test_earned"
]

for c in cols_to_zero_neg:
    final_result_sse_df_test = final_result_sse_df_test.withColumn(c, f.when(f.col(c) < 0, 0).otherwise(f.col(c)))

final_result_sse_df_test.display()

In [0]:
# Processing REM SSE PUSH XCM campaigns + automation
final_result_rem_sse_push_xcm = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_rem_sse_push_xcm') \
    .dropDuplicates() \
    .filter(f.col("effective_date") >= "20240101") \
    .withColumnRenamed("abs_sales_uplift_earned", "abs_sales_uplift") \
    .withColumnRenamed("KPM_PROJECT_ID", "kpm_duplicated_id") \
    .withColumnRenamed("redemption_cost", "new_redemption_cost") \
    .select(
        "kpm_duplicated_id", "working_cost", "new_redemption_cost", "adj_sales_uplift", "abs_sales_uplift", "abs_sales_test_earned", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost"
    )

# Make negative #s to 0, since we do not report negative #s.
cols_to_zero_neg = [
    "working_cost", "adj_sales_uplift", "abs_sales_uplift", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost", "abs_sales_test_earned"
]

for c in cols_to_zero_neg:
    final_result_rem_sse_push_xcm = final_result_rem_sse_push_xcm.withColumn(c, f.when(f.col(c) < 0, 0).otherwise(f.col(c)))
final_result_rem_sse_push_xcm.display()

In [0]:
# Processing XCM and DISP standalone campaigns + automation
final_result_xcm_disp_df_test = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_xcm_disp') \
    .dropDuplicates() \
    .withColumnRenamed("new_sales_uplift_earned", "abs_sales_uplift") \
    .withColumnRenamed("new_sales_test_earned", "abs_sales_test_earned") \
    .withColumnRenamed("redemption_cost", "new_redemption_cost") \
    .select(
        "kpm_duplicated_id", "working_cost", "new_redemption_cost", "adj_sales_uplift", "abs_sales_uplift", "abs_sales_test_earned", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost"
    )

# Overwrite values for kpm_duplicated_id == 136421 (Hard coding this campaign since it has no barcodes for now, fix later)
final_result_xcm_disp_df_test = final_result_xcm_disp_df_test.withColumn(
    "abs_sales_uplift",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(2813986.5)).otherwise(f.col("abs_sales_uplift"))
).withColumn(
    "adj_iroas",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(16.1)).otherwise(f.col("adj_iroas"))
).withColumn(
    "abs_iroas",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(30.4)).otherwise(f.col("abs_iroas"))
).withColumn(
    "adj_aroas",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(85.0)).otherwise(f.col("adj_aroas"))
).withColumn(
    "abs_aroas",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(161.0)).otherwise(f.col("abs_aroas"))
).withColumn(
    "abs_total_cost",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(92700)).otherwise(f.col("abs_total_cost"))
).withColumn(
    "adj_total_cost",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(175000)).otherwise(f.col("adj_total_cost"))
).withColumn(
    "new_redemption_cost",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(0)).otherwise(f.col("new_redemption_cost"))
).withColumn(
    "abs_sales_test_earned",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(14913202)).otherwise(f.col("abs_sales_test_earned"))
)

# Overwrite values for kpm_duplicated_id == 147400 (Hard coding this campaign since it has no barcodes for now, fix later)
final_result_xcm_disp_df_test = final_result_xcm_disp_df_test.withColumn(
    "abs_sales_uplift",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(1718853.8)).otherwise(f.col("abs_sales_uplift"))
).withColumn(
    "adj_iroas",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(9.0)).otherwise(f.col("adj_iroas"))
).withColumn(
    "abs_iroas",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(17.5)).otherwise(f.col("abs_iroas"))
).withColumn(
    "adj_aroas",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(109.0)).otherwise(f.col("adj_aroas"))
).withColumn(
    "abs_aroas",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(210.0)).otherwise(f.col("abs_aroas"))
).withColumn(
    "abs_total_cost",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(97979)).otherwise(f.col("abs_total_cost"))
).withColumn(
    "adj_total_cost",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(190000)).otherwise(f.col("adj_total_cost"))
).withColumn(
    "new_redemption_cost",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(0)).otherwise(f.col("new_redemption_cost"))
).withColumn(
    "abs_sales_test_earned",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(20616986)).otherwise(f.col("abs_sales_test_earned"))
)

# Make negative #s to 0, since we do not report negative #s.
cols_to_zero_neg = [
    "working_cost", "adj_sales_uplift", "abs_sales_uplift", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost", "abs_sales_test_earned"
]

for c in cols_to_zero_neg:
    final_result_xcm_disp_df_test = final_result_xcm_disp_df_test.withColumn(c, f.when(f.col(c) < 0, 0).otherwise(f.col(c)))

# Remove row where kpm_duplicated_id == 131511 and abs_iroas == 0
final_result_xcm_disp_df_test = final_result_xcm_disp_df_test.filter(~((f.col("kpm_duplicated_id") == 131511) & (f.col("abs_iroas") == 0)))

#136421, 147400
final_result_xcm_disp_df_test.display()

In [0]:
# Processing TDC campaigns + automation
final_result_tdc_df_test = spark.read.parquet(
  f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_tdc'
).dropDuplicates() \
 .withColumnRenamed("abs_sales_uplift_earned", "abs_sales_uplift") \
 .withColumnRenamed("redemption_cost", "new_redemption_cost") \
 .withColumn("adj_sales_total", f.col("adj_sales_total").cast("double")) \
 .select(
    "kpm_duplicated_id", "working_cost", "new_redemption_cost", "adj_sales_uplift", "abs_sales_uplift", "abs_sales_test_earned", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost"
)

# Make negative #s to 0, since we do not report negative #s.
cols_to_zero_neg = [
    "working_cost", "adj_sales_uplift", "abs_sales_uplift", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost", "abs_sales_test_earned"
]

for c in cols_to_zero_neg:
    final_result_tdc_df_test = final_result_tdc_df_test.withColumn(c, f.when(f.col(c) < 0, 0).otherwise(f.col(c)))

final_result_tdc_df_test.display()

In [0]:
# Combine TDC, XCM, SSE ABS numbers into one dataframe and write it out
tdc_sse_xcm_disp_abs_df = final_result_tdc_df_test.union(final_result_xcm_disp_df_test).union(
  final_result_sse_df_test).union(final_result_rem_sse_push_xcm)
tdc_sse_xcm_disp_abs_df.display()

tdc_sse_xcm_disp_abs_df.coalesce(1).write.mode("overwrite").parquet('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_all_camp_types')